# TALSIM .PRO File Editor — Fixed-Width Column Patcher

**Purpose:** Programmatically edits specific fields inside a TALSIM `.PRO`
project file without breaking the fixed-width column format, then writes the
patched file to `_edited.PRO`.

**What it does:**
- Defines column byte positions for `[KPRO]` and `[EZG]` blocks
- Applies configurable edit rules (e.g. change `dtE` to 200, `dt` to 60,
  `NID` to 2, `SimStart` to `01.07.2027`, `ETp` to 1)
- Supports optional conditional matching (`when` key) to only patch
  rows that match specific field values
- Preserves all other characters in each line exactly as-is

**Input:** `Neuer_Teich_nr.PRO`  
**Output:** `Neuer_Teich_nr_edited.PRO`

---

In [ ]:
from pathlib import Path

INPUT_FILE = r"C:\Users\raah\Desktop\Project_ZR\Neuer_Teich_nr\Neuer_Teich_nr.PRO"
OUTPUT_FILE = INPUT_FILE.replace(".PRO", "_edited.PRO")

# Fixed-width column positions (start, end)
COLUMNS = {
    "[KPRO]": {
        "dtE": (22, 26),
        "dt":  (29, 31),
        "NID": (65, 66),
        "ID":(5,7),
        "SimStart":(32,42),
    },
    "[EZG]": {
        "Pro_ID": (5, 7),
        "Start":  (31, 33),
        "ETp":    (42, 47),
    },
}

# Editing rules
EDIT_RULES = {
    "[KPRO]": [
        {"column": "dtE",  "new": "200"},
        {"column": "dt",  "new": "60"},
        {"column": "NID", "new": "2"},
        {"column": "SimStart", "new": "01.07.2027"},
    ],
    "[EZG]": [
        {
            "column": "Start",
            "new": "6",
            
        },
        {
            "column": "ETp",
            "new": "1",
        },
    ],
}


def apply_rules_to_line(line, block):
    """Apply edit rules to a single fixed-width line."""
    if block not in EDIT_RULES or block not in COLUMNS:
        return line

    for rule in EDIT_RULES[block]:

        # ---- conditional match (supports sets) ----
        if "when" in rule:
            matched = True
            for col, expected in rule["when"].items():
                cell = line[
                    COLUMNS[block][col][0] : COLUMNS[block][col][1]
                ].strip()

                if isinstance(expected, (set, list, tuple)):
                    if cell not in expected:
                        matched = False
                        break
                else:
                    if cell != expected:
                        matched = False
                        break

            if not matched:
                continue
        # -------------------------------------------

        col = rule["column"]
        start, end = COLUMNS[block][col]

        value = str(rule["new"]).rjust(end - start)
        line = line[:start] + value + line[end:]

    return line


def process_file():
    current_block = None
    output_lines = []

    with open(INPUT_FILE, "r", encoding="latin-1") as f:
        for line in f:
            stripped = line.strip()

            # Detect block header
            if stripped.startswith("[") and stripped.endswith("]"):
                current_block = stripped
                output_lines.append(line)
                continue

            # Only edit data rows
            if current_block and line.startswith(" |"):
                line = apply_rules_to_line(line, current_block)

            output_lines.append(line)

    with open(OUTPUT_FILE, "w", encoding="latin-1") as f:
        f.writelines(output_lines)


process_file()
print("Done ✔")